# 08 — DOE analysis of both factorials (NIST sequence)

Study 1 (GBSA physics) and Study 2 (GROMACS production) are both designed experiments. NBs 05 to 08 analysed them with variance decomposition and pairwise tests instead of the standard DOE (design-of-experiments) toolkit. This NB runs the NIST/SEMATECH Engineering Statistics Handbook sequence (§5.6.1, Eddy Current Probe Sensitivity):

1. **Ordered data plot** — responses sorted; which settings sit at the good end.
2. **DOE scatter plot** — every observation, by factor level.
3. **DOE mean plot** — main effects (level means).
4. **Interaction effects matrix** — every factor pair.
5. **Block plot** — main effects blocked on target; the robustness check.
6. **Estimated effects** — ordered |effect|, with a significance reference.

**Where this NB sits in the reading order.** Two places. §2 (Study 1) needs only the raw factorial and reads best right after NB 05. §3 (Study 2) cites this NB's coverage gate, reproducibility floor (§3.2) and selection rule (§4), so it must follow. Both halves live here so the two factorials get identical treatment and can be compared step by step. Cost: this ambiguity. Stated, not hidden.

**Two departures from the NIST examples, both deliberate.**

- NIST uses 2-level factors, where an effect is a difference of two means. Ours are multi-level (Study 1: 4×3×2×2; Study 2: five 3-level factors in an L27 — orthogonal 27-run array). We generalise with the **range of the level means**, `max(mean) − min(mean)`. Reduces to the usual definition at two levels. So the **half-normal probability plot of effects does not apply** unmodified (it assumes iid 2-level contrasts). Step 6 uses the ordered |effect| plot plus a permutation reference. **Each effect is judged against its own null** — raw effect against a raw permuted null, target-blocked effect against a null that is permuted then blocked identically. An earlier version drew one raw null across both bars. Referee finding, correct: blocking strips between-target variance out of the observed effect but not the null, so the blocked bar was compared against an inflated reference.
- The **block plot (step 5) is load-bearing.** Target explains 70 to 98% of response variance (NB 05), so an unblocked main effect can be an artefact of which targets happen to sit at a level. A factor is credible only if the effect holds within target blocks.

**Preliminary — Study 2 BEDROC section.** Only 99 of 243 (config × target) cells pass the n ≥ 8 gate.


> **Reader guide.** *Experiment A2:* NIST-sequence DOE cross-check of both GBSA factorials.
>
> **Question:** *do the DOE main-effect estimates confirm the η² decomposition of NB20, or does
> a hidden interaction term matter?*
>
> **Method:** full-factorial DOE with the NIST DOE toolkit; per-target main effects + top-2
> interactions.
>
> **Reproducibility contract:** reads `data/raw/reference/ohds_gbsa_dG_raw.csv`; DOE-table
> outputs to `data/derived/doe/`.

In [ ]:
NB_STEM = "21_doe_analysis"
import sys, re, itertools
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.axes import Axes
from matplotlib.figure import Figure
from IPython.display import display

_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / "pyproject.toml").is_file()), _here)
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from gbsabench.paths import RAW, DERIVED, FIGURES
from gbsabench import style, metrics
from gbsabench.io import load
style.apply_style()
NAVY, GOLD, GREY, GREYD = style.NAVY, style.GOLD, style.GREY, style.GREY_DASH
GREY_DASH = GREYD  # alias — some cells use raw style-module name
CREAM = style.CREAM          # panel labels are drawn on a cream plate, see panel()

# repo convention (notebooks 01-07): in-figure titles suppressed, markdown carries
# the description; panel labels go in as in-axes text.
_SUPPRESSED_TITLES = []
def _capture_title(self, *a, **k):
    if a and isinstance(a[0], str) and a[0].strip():
        _SUPPRESSED_TITLES.append((NB_STEM, a[0].strip()))
def _capture_suptitle(self, *a, **k):
    if a and isinstance(a[0], str) and a[0].strip():
        _SUPPRESSED_TITLES.append((NB_STEM, a[0].strip()))
Axes.set_title = _capture_title
Figure.suptitle = _capture_suptitle


def panel(ax, text, dy=None):
    """Panel/row label above the axes, excluded from the layout and never clipped.

    Four revisions, each fixing the previous one's side effect, and all four are recorded
    because the failure mode kept moving rather than going away:
      1. an ordinary in-layout text artist above the axes -- tight_layout reserved column
         width for the string, and "(provisional)" cost 3.9x panel width: 43 px panels
         beside 366 px ones;
      2. `set_in_layout(False)` on that artist -- panels came back to full width, but the
         label then overhung into nothing and the top row's title was clipped by the figure
         edge while the second row's collided with the first row's x-labels;
      3. drawn INSIDE the axes instead -- which reintroduced (1) exactly, because
         tight_layout measures a Text artist's bounding box wherever it sits, and a label
         wider than its panel shrinks the panel just the same;
      4. this one: above the axes as an offset annotation, `annotation_clip=False` so it is
         never cut, and `set_in_layout(False)` so it reserves nothing. The caller must leave
         top margin -- every call site here passes `rect=` to tight_layout.
    """
    _t = ax.annotate(text, xy=(0.0, 1.0), xycoords="axes fraction",
                     xytext=(0, 6), textcoords="offset points",
                     ha="left", va="bottom", fontsize=10.5, weight="bold",
                     color=NAVY, annotation_clip=False, zorder=10)
    _t.set_in_layout(False)
    return _t
FIG_DIR = FIGURES / "doe"; FIG_DIR.mkdir(parents=True, exist_ok=True)
DER_DIR = DERIVED / "doe"; DER_DIR.mkdir(parents=True, exist_ok=True)
# Two SEPARATE streams. They used to be one, so how many jitter draws a plot cell
# consumed changed the permutation p-values of every cell run after it -- the DOE
# p's depended on execution order, not just on the seed (referee finding, round 2).
RNG_JITTER = np.random.default_rng(20260828)      # cosmetic only: point jitter
RNG_PERM   = np.random.default_rng(20260828)      # inferential: permutation nulls
RNG = RNG_JITTER                                  # back-compat for plotting helpers
print("setup ok")

## 1. The six DOE steps, as reusable functions

Written once and applied to both factorials, so the two studies get identical treatment and can be compared step by step.


In [ ]:
def level_means(df, factor, response):
    return df.groupby(factor)[response].mean()


def effect_range(df, factor, response):
    """Generalised effect: range of level means (= |difference| at 2 levels)."""
    m = level_means(df, factor, response)
    return float(m.max() - m.min())


def blocked_effect(df, factor, response, block="target"):
    """Effect after removing block means -- immune to target composition."""
    d = df.copy()
    # `block` may be a list: the L27 needs target AND dt, because the realised array is
    # not balanced on dt and dt is aliased with two of the other factors.
    d[response] = d[response] - d.groupby(list(block) if isinstance(block, (list, tuple))
                                          else block)[response].transform("mean")
    return effect_range(d, factor, response)


def ordered_data_plot(ax, df, response, factors, top_n=None):
    d = df.sort_values(response).reset_index(drop=True)
    if top_n and len(d) > top_n:                      # keep both tails, drop middle
        d = pd.concat([d.head(top_n // 2), d.tail(top_n // 2)]).reset_index(drop=True)
    ax.plot(range(len(d)), d[response], "o-", color=NAVY, ms=3, lw=0.8)
    ax.set_xlabel("observation (sorted by response)")
    ax.set_ylabel(response)
    return d


def doe_scatter(axes, df, response, factors):
    for ax, f in zip(axes, factors):
        lv = sorted(df[f].unique())
        for i, l in enumerate(lv):
            y = df[df[f] == l][response].to_numpy()
            x = np.full(len(y), i) + RNG.normal(0, 0.06, len(y))
            ax.plot(x, y, "o", color=GREY, ms=2.5, alpha=0.5)
        ax.set_xticks(range(len(lv))); ax.set_xticklabels(lv, fontsize=8)
        ax.set_xlabel(f, fontsize=9)


def doe_mean(axes, df, response, factors, block=None):
    grand = df[response].mean()
    for ax, f in zip(axes, factors):
        m = level_means(df, f, response)
        ax.plot(range(len(m)), m.values, "o-", color=NAVY, ms=6, lw=1.6)
        ax.axhline(grand, color=GREYD, ls="--", lw=1)
        ax.set_xticks(range(len(m))); ax.set_xticklabels(m.index, fontsize=8)
        ax.set_xlabel(f, fontsize=9)


def interaction_matrix(fig, gs, df, response, factors):
    """Lower-triangle matrix: mean response vs factor A level, one line per B level."""
    n = len(factors)
    for i, fa in enumerate(factors):
        for j, fb in enumerate(factors):
            if j >= i:
                continue
            ax = fig.add_subplot(gs[i, j])
            piv = df.groupby([fa, fb])[response].mean().unstack()
            for k, col in enumerate(piv.columns):
                ax.plot(range(len(piv.index)), piv[col].values, "o-", ms=3.5, lw=1.2,
                        color=plt.cm.cividis(k / max(1, len(piv.columns) - 1)),
                        label=f"{fb}={col}")
            ax.set_xticks(range(len(piv.index)))
            ax.set_xticklabels(piv.index, fontsize=6.5)
            ax.tick_params(labelsize=6.5)
            # x carries fa's levels (piv.index); the lines are fb's levels.
            if j == 0:
                ax.set_ylabel(f"mean {response}", fontsize=8)
            ax.set_xlabel(fa, fontsize=8)
            if i == 1 and j == 0:
                ax.legend(fontsize=5.5, frameon=False, ncol=2)


def block_plot(axes, df, response, factors, block="target"):
    """Per-block level means: does the effect hold WITHIN every block?"""
    blocks = sorted(df[block].unique())
    for ax, f in zip(axes, factors):
        lv = sorted(df[f].unique())
        for b in blocks:
            m = level_means(df[df[block] == b], f, response).reindex(lv)
            ax.plot(range(len(lv)), m.values, "-", color=GREY, lw=0.9, alpha=0.65)
        m = level_means(df, f, response).reindex(lv)
        ax.plot(range(len(lv)), m.values, "o-", color=NAVY, lw=2.2, ms=6)
        ax.set_xticks(range(len(lv))); ax.set_xticklabels(lv, fontsize=8)
        ax.set_xlabel(f, fontsize=9)


def effects_table(df, response, factors, block="target", n_perm=20000,
                  unit=None, strata=None):
    """`unit` names the column the factor is ASSIGNED to (e.g. "config").

    THE EXCHANGEABILITY UNIT. Study 2's factors are properties of a CONFIG -- 27
    independent assignments -- but the frame has 3 115 runs, and the null was built by
    shuffling the factor across rows. That treats ~115 correlated runs as independent and
    inflates the null by ~7x: q95 goes 13 s -> 93 s at the config unit, and rcoulomb, gap
    and nstlist all fall from p < 0.02 to p = 0.83-0.94. Only MTS survives. Two referees
    found this independently (round 9) and it predates every other change to this cell.

    Study 1 is fully crossed so it is largely immune -- with one exception the referees
    also found: `igb` on BEDROC moves 0.027 -> 0.138 at the combo unit, and it is the
    first example this notebook cites for "blocking reveals a real effect".

    TWO nulls are built, because two effects are reported. The raw effect is referenced
    against a raw permuted null; the target-blocked effect is referenced against a null
    that is permuted *and then blocked the same way*. Comparing a blocked effect to an
    unblocked null is not a valid test -- blocking strips between-target variance out of
    the observed effect but not out of the null, so the blocked bar is judged against a
    reference that is systematically too large.

    (This paragraph was the function's original docstring; an insertion left it stranded
    as a bare string expression below the new one, so `help()` no longer showed the
    sentence explaining the function's central design decision. Referee, round 11.)
    """
    rows = []
    _degenerate = set()
    # `block` may be a list of columns (the L27 needs target AND dt). `x in df` tests
    # column membership for a string and raises/misfires for a list, so normalise first.
    _blk_cols = list(block) if isinstance(block, (list, tuple)) else [block]
    has_block = all(_c in df.columns for _c in _blk_cols)
    for f in factors:
        obs = effect_range(df, f, response)
        blk = blocked_effect(df, f, response, block) if has_block else np.nan
        d = df.copy()
        null_raw, null_blk = [], []
        # THE PERMUTATION GROUP, not just the unit. Round 9 fixed the unit (config, not
        # run) and left the group wrong: it permuted the assignment FREELY across units.
        # In a crossed design that lets the permuted factor align with a DIFFERENT factor
        # and drag its variance into the null. It cost a real result -- Study 1's `igb`
        # was retracted at p = 0.13 when the design-respecting null gives p < 1e-4,
        # because free permutation let `igb` align with `intdiel`, whose effect is 2.4x
        # larger. Two referees found it independently (round 11).
        #
        # `strata` names the other design columns. The assignment is permuted only WITHIN
        # each stratum, which is the randomisation the design actually supports.
        _design = (df[[unit] + [x for x in factors]].drop_duplicates(unit).set_index(unit)
                   if unit and unit in df.columns else None)
        _assign = _design[f] if _design is not None else None
        _strata = [x for x in (strata or []) if x != f and _design is not None
                   and x in _design.columns]
        # A stratified permutation only EXISTS if some stratum holds more than one unit.
        # On the L27, stratifying a factor on the other four leaves 27 strata of size 1 --
        # the identity. Every p would be 1.0 and every null would equal the observed
        # effect. That is not a conservative test, it is NO test: this design supports no
        # exact randomisation of a single factor. A referee found it by running our own
        # code path (round 11). Detected and refused rather than reported.
        if _strata and _design.groupby(_strata).size().max() < 2:
            _degenerate.add(f)
            rows.append({"factor": f, "effect": obs, "effect_blocked": blk,
                         "null_q95": np.nan, "p_perm": np.nan,
                         "null_q95_blocked": np.nan, "p_perm_blocked": np.nan})
            continue
        for _ in range(n_perm):
            if _assign is not None:
                if _strata:
                    _a = _design.copy()
                    _a[f] = (_a.groupby(_strata)[f]
                             .transform(lambda v: RNG_PERM.permutation(v.to_numpy())))
                    d[f] = df[unit].map(_a[f].to_dict())
                else:
                    _perm = dict(zip(_assign.index, RNG_PERM.permutation(_assign.to_numpy())))
                    d[f] = df[unit].map(_perm)
            else:
                d[f] = RNG_PERM.permutation(d[f].to_numpy())
            null_raw.append(effect_range(d, f, response))
            if has_block:
                null_blk.append(blocked_effect(d, f, response, block))
        null_raw = np.asarray(null_raw)
        row = {"factor": f, "effect": obs, "effect_blocked": blk,
               "null_q95": np.percentile(null_raw, 95),
               "p_perm": (np.sum(null_raw >= obs) + 1) / (n_perm + 1)}
        if has_block:
            null_blk = np.asarray(null_blk)
            row["null_q95_blocked"] = np.percentile(null_blk, 95)
            row["p_perm_blocked"] = (np.sum(null_blk >= blk) + 1) / (n_perm + 1)
        else:
            row["null_q95_blocked"] = np.nan
            row["p_perm_blocked"] = np.nan
        rows.append(row)
    out = pd.DataFrame(rows).sort_values("effect", ascending=False).reset_index(drop=True)
    out["no_exact_test"] = out.factor.isin(_degenerate)
    # The unit and the reference set ARE the result: the same effect column carries three
    # different verdicts depending on them, and three superseded rounds of Study-2
    # significance claims came from changing them silently. They travel with the CSV so a
    # reader of the table cannot lose them, and verify.py asserts them (referee, round 11).
    out["perm_unit"] = unit or "row"
    out["perm_strata"] = "+".join(strata) if strata else "none (free)"
    out["perm_block"] = "+".join(_blk_cols) if has_block else "none"
    out["n_perm"] = n_perm
    if _degenerate:
        print(f"    NOTE: no design-respecting permutation exists for {sorted(_degenerate)}"
              f" -- every stratum is a singleton. p-values left BLANK rather than reporting"
              f" an identity permutation (p=1.0) or a free one (known wrong). Judge these"
              f" against the measured repeat spread in section 3.8.")
    return out


GOLD_DASH = "#5A4A08"   # dark gold: reference line belonging to the gold bar


def effects_plot(ax, tab, blocked_on="target", not_estimable=()):
    """Each bar is drawn against ITS OWN permuted null -- raw vs raw null,
    blocked vs blocked null. One shared line would misrepresent the blocked bar
    (see effects_table)."""
    # Bars are placed with a GAP between factor groups. Previously the two bars of a
    # factor and the bars of the next factor were all 0.45 apart in a 3.6in canvas, so
    # eight bars touched and the grouping was invisible (supervisor, round 7).
    y = np.arange(len(tab))[::-1] * 1.35
    has_blk = tab["effect_blocked"].notna().any()
    ax.barh(y, tab["effect"], color=NAVY, height=0.42, label="|effect| (raw)")
    for yy, q in zip(y, tab["null_q95"]):
        ax.plot([q, q], [yy - 0.21, yy + 0.21], color=GREYD, lw=1.8, ls="--")
    if has_blk:
        ax.barh(y - 0.44, tab["effect_blocked"], color=GOLD, height=0.42,
                label=f"|effect| (blocked on {blocked_on})")
        for yy, q in zip(y, tab["null_q95_blocked"]):
            ax.plot([q, q], [yy - 0.65, yy - 0.23], color=GOLD_DASH, lw=1.8, ls=":")
    ax.plot([], [], color=GREYD, lw=1.8, ls="--", label="95th pct, raw null")
    if has_blk:
        ax.plot([], [], color=GOLD_DASH, lw=1.8, ls=":", label="95th pct, blocked null")
    # Mark bars that are not estimable rather than drawing them as zero. Blocking on dt
    # removes dt's own effect (its blocked bar is ~1e-14 and rendered as nothing, which a
    # figure-only reader reads as "the timestep does not matter" -- the opposite of the
    # result), and conditional on dt the gap and nstlist levels determine each other, so
    # their two blocked effects are ONE contrast counted twice (referees, round 9).
    # The gate was `abs(effect) < 1e-9`, which only ever fired for the blocked-out factor
    # itself -- so `gap` and `nstlist` were drawn as ordinary bars while three documents
    # said they were marked. Three referees caught it twice. Aliasing is a property of the
    # DESIGN, so it is passed in explicitly rather than inferred from a magnitude.
    # Annotate on a MISSING blocked effect, not on a small one. The gate used to be
    # `abs(effect_blocked) < 1e-9`, which fires only for the blocked-out factor itself, so
    # `gap` and `nstlist` were drawn as ordinary gold bars while the notebook, the figure
    # caption and the response letter all said they were marked. Four referees measured
    # that on the shipped PNG across three rounds. Estimability is a property of the
    # DESIGN, so it is now written into the table as NaN by `mark_not_estimable` and the
    # figure simply draws what the table says.
    _why = tab["blocked_note"] if "blocked_note" in tab.columns else pd.Series("", index=tab.index)
    for yy, fac, val, note in zip(y, tab["factor"], tab["effect_blocked"], _why):
        if fac in not_estimable or pd.isna(val) or abs(val) < 1e-9:
            ax.text(ax.get_xlim()[1]*0.02, yy - 0.44,
                    note or "not estimable under this blocking",
                    va="center", fontsize=7.5, color=GOLD_DASH, style="italic")
    ax.set_yticks(y - 0.22); ax.set_yticklabels(tab["factor"])
    ax.margins(y=0.10)
    ax.set_xlabel("effect size (range of level means)")
    # Legend OUTSIDE the axes: it was overlapping the two smallest bars.
    ax.legend(fontsize=8, frameon=False, loc="upper left",
              bbox_to_anchor=(1.01, 1.0), borderaxespad=0)

print("DOE helpers defined")

def mark_not_estimable(tab, notes):
    """Blank the BLOCKED columns of factors whose blocked contrast is not their own.

    `notes` maps factor -> the reason, which is printed on the figure. Nothing is
    collapsed and nothing is renamed: the raw (unblocked) column stays, because raw
    `gap` (47.9 s) and raw `nstlist` (71.9 s) are genuinely different marginal contrasts
    and the table's own numbers refute calling them "one contrast counted twice".

    What IS true, and what this marks: the blocked column conditions on dt, and *within
    each dt stratum the gap and nstlist partitions of the 9 configs coincide*. So each
    factor's blocked main effect is aliased with the other's dt interaction, and neither
    blocked number can be attributed to its own factor. Referees supplied this wording
    after two rounds of the authors' looser version.
    """
    t = tab.copy()
    if "blocked_note" not in t.columns:
        t["blocked_note"] = ""
    for fac, why in notes.items():
        m = t.factor == fac
        if not m.any():
            continue
        t.loc[m, ["effect_blocked", "null_q95_blocked", "p_perm_blocked"]] = np.nan
        t.loc[m, "blocked_note"] = why
    return t


def pfmt(p, n_perm=20000):
    """Render a permutation p, never as 0. The attainable floor is 1/(n_perm+1), so a
    p at the floor is a statement about the draw count, not about the evidence, and
    printing it as `0.0000` says something the test cannot support (referees, 4 rounds)."""
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    floor = 1.0 / (n_perm + 1)
    return f"< {floor:.1e} (floor)" if p <= floor + 1e-12 else f"{p:.4f}"


def anova_config_means(cfg, response, factors):
    """Least-squares ANOVA on the 27 CONFIG means -- the design's own unit.

    A cross-check on the permutation test, suggested by a referee: the L27's design matrix
    with five three-level factors has rank 11 with 16 residual degrees of freedom, so all
    five main effects ARE estimable by least squares even where an exact randomisation of a
    single factor is not available. Reported alongside the permutation p, not instead of it.
    """
    import itertools as _it
    from scipy import stats as _st
    d = cfg.dropna(subset=[response])
    X = [np.ones(len(d))]
    names = []
    for f in factors:                      # dummy code: 2 columns per 3-level factor
        lv = sorted(d[f].unique())
        for l in lv[1:]:
            X.append((d[f] == l).to_numpy(float)); names.append(f"{f}={l}")
    X = np.column_stack(X)
    y = d[response].to_numpy(float)
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    rss_full = float(((y - X @ beta) ** 2).sum())
    df_res = len(y) - np.linalg.matrix_rank(X)
    rows = []
    for f in factors:
        keep = [0] + [i + 1 for i, n in enumerate(names) if not n.startswith(f + "=")]
        Xr = X[:, keep]
        br, *_ = np.linalg.lstsq(Xr, y, rcond=None)
        rss_red = float(((y - Xr @ br) ** 2).sum())
        df_f = X.shape[1] - Xr.shape[1]
        F = ((rss_red - rss_full) / df_f) / (rss_full / df_res) if df_res > 0 else np.nan
        rows.append({"factor": f, "df": df_f, "F": F,
                     "p_anova": float(_st.f.sf(F, df_f, df_res)) if df_res > 0 else np.nan})
    out = pd.DataFrame(rows)
    out.attrs["df_res"] = int(df_res)
    out.attrs["rank"] = int(np.linalg.matrix_rank(X))
    return out


## 2. Study 1 — GBSA physics factorial

Design: `igb{1,2,5,8}` × `intdiel{1,2,4}` × `saltcon{0,0.15}` × `surften{0.0072,0}` = 48 combos. Each applied post-hoc to the same trajectory per complex, so all 48 share one ensemble.

Response: **BEDROC α=20** per (target, combo) — the primary metric. Kendall τ carried alongside.


In [ ]:
gbsa = load("gbsa_dG_raw"); meta = load("metadata")
merged = gbsa.merge(meta, on=["complex_id", "target"])


def parse_combo(c):
    igb, di, salt, st = re.match(r"igb(\d+)_di(\d+)_salt([0-9.]+)_st([0-9.]+)", c).groups()
    return {"igb": igb, "intdiel": di, "saltcon": salt, "surften": st}


rows = []
for (t, c), lig in merged.groupby(["target", "combo"]):
    lig = lig.dropna(subset=["pchembl", "is_active", "mean_dG_kcalmol"])
    lab = lig.is_active.astype(int).to_numpy()
    if lab.sum() in (0, len(lab)):
        continue
    r = {"target": t, "combo": c,
         "bedroc": metrics.bedroc(-lig.mean_dG_kcalmol.to_numpy(), lab),
         "tau": metrics.kendall_tau(lig.mean_dG_kcalmol.to_numpy(), lig.pchembl.to_numpy())}
    r.update(parse_combo(c)); rows.append(r)

S1 = pd.DataFrame(rows)
F1 = ["igb", "intdiel", "saltcon", "surften"]
print(f"Study 1: {len(S1)} (target,combo) observations, {S1.target.nunique()} targets")
display(S1.head(3))

### 2.1 Ordered data plot & 2.2 DOE scatter plot


In [ ]:
fig = plt.figure(figsize=(13, 4.2))
gs = fig.add_gridspec(1, 5, width_ratios=[1.5, 1, 1, 1, 1], wspace=0.32)
ax0 = fig.add_subplot(gs[0, 0])
ordered_data_plot(ax0, S1, "bedroc", F1)
panel(ax0, "(A) Ordered data")
axs = [fig.add_subplot(gs[0, i]) for i in range(1, 5)]
doe_scatter(axs, S1, "bedroc", F1)
panel(axs[0], "(B) DOE scatter — every (target,combo) observation")
axs[0].set_ylabel("BEDROC (α=20)")
for a in axs[1:]:
    a.set_yticklabels([])
# rect= reserves top margin for panel()'s label, which is drawn above the axes and
# excluded from the layout, so tight_layout would otherwise leave it no room.
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig(FIG_DIR / f"{NB_STEM}_s1_ordered_scatter.png", dpi=170)
plt.savefig(FIG_DIR / f"{NB_STEM}_s1_ordered_scatter.pdf")
plt.show()

### 2.2b DOE scatter, flat — every factor's every level on one X-axis

The four-panel dexsplot in 2.2 is the NIST default (one panel per factor). Concatenated form puts every factor's levels along a single X-axis: every observation is a dot, level-mean is a bold navy square, grand mean is a gold reference line. If a factor is flat against the reference, it does not shift BEDROC on its own.

Same data as 2.2, same jitter seed, just re-laid out.


In [ ]:
# 2.2b — flat DOE scatter across all four factors in one panel
fig, ax = plt.subplots(figsize=(12, 5.5))
_RNG = np.random.default_rng(20260902)
_x_offset, _gap = 0, 1.2
_tick_pos, _tick_lab, _bands = [], [], []
for f in F1:
    _levels = sorted(S1[f].unique())
    for i, lv in enumerate(_levels):
        _x = _x_offset + i
        _y = S1[S1[f] == lv]["bedroc"].to_numpy()
        _xj = np.full(len(_y), _x) + _RNG.normal(0, 0.09, len(_y))
        ax.plot(_xj, _y, "o", color=GREY, ms=3, alpha=0.55, zorder=1)
        ax.plot([_x], [_y.mean()], "s", color=NAVY, ms=10, mec="white", mew=1, zorder=5)
        _tick_pos.append(_x); _tick_lab.append(str(lv))
    _bands.append((_x_offset - 0.4, _x_offset + len(_levels) - 1 + 0.4, f, _x_offset + (len(_levels) - 1)/2))
    _x_offset += len(_levels) + _gap

for j in range(len(_bands) - 1):
    _r = _bands[j][1]; _l = _bands[j+1][0]
    ax.axvline((_r + _l) / 2, color=GREY_DASH, ls=":", lw=0.9, zorder=0)

_grand = S1["bedroc"].mean()
ax.axhline(_grand, color=GOLD, ls="--", lw=1.2, zorder=0, label=f"grand mean = {_grand:.3f}")

ax.set_xticks(_tick_pos); ax.set_xticklabels(_tick_lab, fontsize=9)
ax.set_xlim(-1, _x_offset - _gap)
_ymin, _ymax = ax.get_ylim()
for _s, _e, _n, _m in _bands:
    ax.text(_m, _ymin - 0.06 * (_ymax - _ymin), _n,
            ha="center", va="top", fontsize=11, fontweight="bold", color=NAVY)

ax.set_ylabel("per-(target, combo) BEDROC (α=20)", fontsize=10)
ax.set_title("Study 1 GBSA — every level of every parameter, one panel  ·  navy square = level mean",
             color=NAVY, fontweight="bold", pad=10, fontsize=11)
ax.legend(loc="upper right", fontsize=9, frameon=False)
ax.grid(axis="y", ls=":", alpha=0.4)
for _sp in ["top", "right"]: ax.spines[_sp].set_visible(False)

plt.tight_layout()
plt.savefig(FIG_DIR / f"{NB_STEM}_s1_dexsplot_flat.png", dpi=300, facecolor=CREAM)
plt.savefig(FIG_DIR / f"{NB_STEM}_s1_dexsplot_flat.pdf", facecolor=CREAM)
plt.show()

### 2.3 DOE mean plot (main effects) & 2.5 block plot

Left row: level means, the classic main-effects read. Right row: same means recomputed **within each target** (grey lines) with the pooled mean over them. If a grey bundle is flat while the navy line slopes, the "effect" is target composition, not physics.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 7), sharey="row")
doe_mean(axes[0], S1, "bedroc", F1)
panel(axes[0][0], "(A) DOE mean plot — main effects on BEDROC")
axes[0][0].set_ylabel("mean BEDROC")
block_plot(axes[1], S1, "bedroc", F1, block="target")
panel(axes[1][0], "(B) Block plot — grey = one target each, navy = pooled")
axes[1][0].set_ylabel("BEDROC by target")
# rect= reserves top margin for panel()'s label, which is drawn above the axes and
# excluded from the layout, so tight_layout would otherwise leave it no room.
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig(FIG_DIR / f"{NB_STEM}_s1_mean_block.png", dpi=170)
plt.savefig(FIG_DIR / f"{NB_STEM}_s1_mean_block.pdf")
plt.show()

### 2.4 Interaction effects matrix


In [ ]:
fig = plt.figure(figsize=(10, 9))
gs = fig.add_gridspec(len(F1), len(F1), hspace=0.45, wspace=0.35)
interaction_matrix(fig, gs, S1, "bedroc", F1)
fig.text(0.005, 0.995, "Interaction effects matrix — BEDROC, Study 1",
         ha="left", va="top", fontsize=11, weight="bold", color=NAVY)
plt.savefig(FIG_DIR / f"{NB_STEM}_s1_interactions.png", dpi=170, bbox_inches="tight")
plt.savefig(FIG_DIR / f"{NB_STEM}_s1_interactions.pdf", bbox_inches="tight")
plt.show()

### 2.6 Estimated effects


In [ ]:
E1 = effects_table(S1, "bedroc", F1, block="target", unit="combo", strata=F1)
E1.round(6).to_csv(DER_DIR / "study1_effects_bedroc.csv", index=False)
display(E1.round(4))
print("permutation p, floor-aware (20 000 draws -> attainable floor 1/20001 = 5.0e-05):")
for _r in E1.itertuples():
    print(f"  {_r.factor:9s} raw {pfmt(_r.p_perm):>18s}   blocked {pfmt(_r.p_perm_blocked):>18s}")
print("`igb` sits AT the floor, so the honest statement is 'detectable at the smallest p this")
print("test can produce', not a p of 5.0e-05 quoted as a measured value. Its effect is 0.065")
print("BEDROC -- see section 3.8 for what that is worth against the measured repeat spread.")

fig, ax = plt.subplots(figsize=(9.6, 5.0))
effects_plot(ax, E1)
panel(ax, "Estimated effects — Study 1, BEDROC")
# rect= reserves top margin for panel()'s label, which is drawn above the axes and
# excluded from the layout, so tight_layout would otherwise leave it no room.
plt.tight_layout(rect=[0, 0, 0.80, 0.94])
plt.savefig(FIG_DIR / f"{NB_STEM}_s1_effects.png", dpi=170)
plt.savefig(FIG_DIR / f"{NB_STEM}_s1_effects.pdf")
plt.show()

E1t = effects_table(S1, "tau", F1, block="target", unit="combo", strata=F1)
E1t.round(6).to_csv(DER_DIR / "study1_effects_tau.csv", index=False)
print("same analysis on Kendall tau:")
display(E1t.round(4))

# 3. Study 2 — GROMACS production parameters (Taguchi L27)

Design: `dt{2,3,4} fs` × `MTS{1,2,3}` × `rcoulomb{1.0,1.1,1.2}` × `gap{0,0.1,0.2}` × `nstlist{20,40,80}`. 27 of 3⁵ runs, main effects orthogonal.

**Both responses carry a coverage caveat, and the wallclock one is bigger than we used to admit.** An earlier revision called the wallclock grid *"the full 3 043-run grid, complete, no coverage caveat"*. Wrong twice over. The cell below counts it.

- **wallclock** — 3 043 of 7 290 possible (config × complex) cells present, i.e. **41.7%**. The missing 58.3% is not scattered. dt = 2 is **complete** (2 430 / 2 430). dt = 3 is **80.2% missing** (481 / 2 430). dt = 4 is **94.6% missing** (132 / 2 430). Coverage is structurally confounded with the study's largest factor — the missing cells are the ones whose MD blew up. That is why every effect below is blocked on `dt` as well as on target: blocking on target alone leaves dt composition inside every other factor's estimate. It also means `dt`'s own raw effect compares three very unequal samples. Separately, the joined frame has 3 115 rows rather than 3 043 because 72 `(config, complex_id)` pairs were retried and both attempts kept — all 72 in dt = 2. §3.8 uses those 72 as the study's only measurement of run-to-run spread.
- **BEDROC** — the rescore campaign, still incomplete. Gated at n ≥ 8 per cell (99 of 243 cells pass). Read only as provisional. `dt = 4` survives on **one target**.

Two responses, different data quality, and neither is "complete".


In [ ]:
man = load("md_variants_manifest_raw")
perf = load("md_variants_prod_perf_raw")
F2 = ["dt_fs", "mts", "rcoulomb", "gap", "nstlist"]

wall = man.merge(perf, left_on=["config", "complex_id"], right_on=["config", "cid"],
                 how="inner", suffixes=("", "_p"))
wall = wall[wall["status"] == "OK"].copy()
wall["wall_s"] = wall["wall_s"].astype(float)
print(f"Study 2 wallclock: {len(wall)} runs, {wall.config.nunique()} configs")
display(wall.groupby("dt_fs").wall_s.describe()[["count", "mean", "std"]].round(1))

### 3.1 Ordered data plot & 3.2 DOE scatter plot — wallclock


In [ ]:
fig = plt.figure(figsize=(15, 4.2))
gs = fig.add_gridspec(1, 6, width_ratios=[1.5, 1, 1, 1, 1, 1], wspace=0.3)
ax0 = fig.add_subplot(gs[0, 0])
# top_n=600 kept head(300)+tail(300) and plotted them contiguously, so the "cliff" at
# x~300 was the excision seam -- ~2500 deleted points -- not a bimodal population. The
# largest true gap in the data is 473 s. Plot all of it (referee finding, round 6/7).
ordered_data_plot(ax0, wall, "wall_s", F2)
panel(ax0, "(A) Ordered data")
ax0.set_ylabel("wallclock (s)")
axs = [fig.add_subplot(gs[0, i]) for i in range(1, 6)]
doe_scatter(axs, wall, "wall_s", F2)
panel(axs[0], f"(B) DOE scatter — all {len(wall):,} OK production runs")
for a in axs[1:]:
    a.set_yticklabels([])
# rect= reserves top margin for panel()'s label, which is drawn above the axes and
# excluded from the layout, so tight_layout would otherwise leave it no room.
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig(FIG_DIR / f"{NB_STEM}_s2_ordered_scatter.png", dpi=170)
plt.savefig(FIG_DIR / f"{NB_STEM}_s2_ordered_scatter.pdf")
plt.show()

### 3.3 DOE mean plot & 3.5 block plot — wallclock

Here the block is the **target** again, because system size drives wallclock (4A5S is ~2× everything else). A parameter effect that survives blocking is a genuine throughput lever, not a system-size artefact.


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 7), sharey="row")
doe_mean(axes[0], wall, "wall_s", F2)
panel(axes[0][0], "(A) DOE mean plot — main effects on wallclock")
axes[0][0].set_ylabel("mean wallclock (s)")
block_plot(axes[1], wall, "wall_s", F2, block="target")
panel(axes[1][0], "(B) Block plot — grey = one target each")
axes[1][0].set_ylabel("wallclock by target (s)")
# rect= reserves top margin for panel()'s label, which is drawn above the axes and
# excluded from the layout, so tight_layout would otherwise leave it no room.
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig(FIG_DIR / f"{NB_STEM}_s2_mean_block.png", dpi=170)
plt.savefig(FIG_DIR / f"{NB_STEM}_s2_mean_block.pdf")
plt.show()

### 3.4 Interaction effects matrix — wallclock


In [ ]:
fig = plt.figure(figsize=(12, 10.5))
gs = fig.add_gridspec(len(F2), len(F2), hspace=0.5, wspace=0.38)
interaction_matrix(fig, gs, wall, "wall_s", F2)
fig.text(0.005, 0.995, "Interaction effects matrix — wallclock, Study 2 (L27)",
         ha="left", va="top", fontsize=11, weight="bold", color=NAVY)
plt.savefig(FIG_DIR / f"{NB_STEM}_s2_interactions.png", dpi=170, bbox_inches="tight")
plt.savefig(FIG_DIR / f"{NB_STEM}_s2_interactions.pdf", bbox_inches="tight")
plt.show()

### 3.6 Estimated effects — wallclock


In [ ]:
# Block on target AND dt. The realised L27 is NOT balanced on dt -- 2502/481/132 rows
# against 1038 if balanced -- and the imbalance concentrates in `gap` and `nstlist`, which
# are dt's alias partners in this resolution-III array. Blocking on target alone therefore
# leaves dt composition inside every other factor's effect: rcoulomb falls 66.0 -> 16.8 s
# (74 % was dt) and gap 59.3 -> 19.4 s (67 %) once dt is blocked. Those two are exactly the
# factors this notebook previously reported as blocking's success case (referee, round 8).
# THE RANDOMISATION SET. Round 11 stratified each factor on the other four; in an L27
# any four factors determine the fifth, so all 27 strata are singletons, the permutation is
# the identity, and every p is exactly 1.0 -- no test at all. A referee measured that on our
# own code path before it shipped. Conditioning on `dt` ALONE (3 strata of 9 configs) is the
# reference set this design does support: it preserves the alias structure that matters --
# dt is confounded with HMR and, within a dt level, with gap/nstlist -- and leaves the other
# factors free to move. dt itself has no stratum to condition on, so it is permuted freely
# across the 27 configs, which is the correct unrestricted test for dt.
S2_STRATA = ["dt_fs"]
# gap and nstlist are ONE contrast once dt is fixed, so they are collapsed for display and
# for the figure. The CSV keeps both rows because both are separately meaningful raw
# (unblocked) marginals, and `no_exact_test`/the figure mark carry the aliasing.
# Within each dt stratum the gap and nstlist partitions of the 9 configs COINCIDE, so each
# one's blocked main effect is aliased with the other's dt interaction. Their blocked columns
# are therefore blanked (NaN) and marked on the figure; the raw columns stay, because
# unblocked they are genuinely different marginal contrasts (47.9 s vs 71.9 s).
S2_ALIAS = {"gap": "aliased with the other's dt interaction — not estimable", "nstlist": "aliased with the other's dt interaction — not estimable",
            "dt_fs": "blocked out by the dt block — read the raw column"}

E2 = effects_table(wall, "wall_s", F2, block=["target", "dt_fs"],
                   n_perm=20000, unit="config", strata=S2_STRATA)
E2 = mark_not_estimable(E2, S2_ALIAS)
E2.round(6).to_csv(DER_DIR / "study2_effects_wallclock.csv", index=False)
display(E2.round(2))
print("permutation p, floor-aware:")
for _r in E2.itertuples():
    print(f"  {_r.factor:9s} raw {pfmt(_r.p_perm):>18s}   blocked {pfmt(_r.p_perm_blocked):>18s}")

# Least-squares cross-check at the design's own unit: the 27 config means. All five main
# effects are estimable here (rank 11, 16 residual df) even where an exact randomisation of
# a single factor is not, so this says whether the permutation verdict is an artefact of the
# reference set. Referee suggestion, round 11.
_cfg_w = wall.groupby(["config"] + F2, as_index=False)["wall_s"].mean()
_av = anova_config_means(_cfg_w, "wall_s", F2)
print(f"\nANOVA on the {len(_cfg_w)} config means "
      f"(design rank {_av.attrs['rank']}, {_av.attrs['df_res']} residual df):")
_av.round(6).to_csv(DER_DIR / "study2_anova_wallclock.csv", index=False)
print(_av.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(9.6, 5.4))
effects_plot(ax, E2, blocked_on="target + dt")
ax.set_xlabel("effect size (range of level means), seconds")
panel(ax, "Estimated effects — Study 2, wallclock")
# rect= reserves top margin for panel()'s label, which is drawn above the axes and
# excluded from the layout, so tight_layout would otherwise leave it no room.
plt.tight_layout(rect=[0, 0, 0.80, 0.94])
plt.savefig(FIG_DIR / f"{NB_STEM}_s2_effects.png", dpi=170)
plt.savefig(FIG_DIR / f"{NB_STEM}_s2_effects.pdf")
plt.show()

### 3.7 The same DOE sequence on BEDROC — provisional

The rescore campaign is incomplete (see NB 08). Gated at n ≥ 8 complexes per (config, target) cell, read as provisional. Included so the production-parameter analysis covers the response that matters, not only the one with complete data.


In [ ]:
sc = load("md_variants_gbsa_scores_raw")
ok = sc[sc.status == "ok"].copy()
ok["delta_total"] = pd.to_numeric(ok["delta_total"], errors="coerce")
ok["is_active"] = ok["is_active"].astype(str).str.lower().map({"true": 1, "false": 0})
ok = ok.dropna(subset=["delta_total", "is_active"])

rows = []
for (c, t), g in ok.groupby(["config", "target"]):
    if len(g) < 8 or g.is_active.nunique() < 2:
        continue
    rows.append({"config": c, "target": t, "n": len(g),
                 "bedroc": metrics.bedroc(g.delta_total.to_numpy(),
                                          g.is_active.to_numpy().astype(int),
                                          alpha=20.0, higher_is_better=False),
                 **{f: g[f].iat[0] for f in F2}})
S2B = pd.DataFrame(rows)
print(f"cells passing the n>=8 gate: {len(S2B)} of 243  "
      f"({S2B.config.nunique()} configs, {S2B.target.nunique()} targets)")

if len(S2B) >= 20:
    fig, axes = plt.subplots(2, 5, figsize=(15, 7), sharey="row")
    doe_mean(axes[0], S2B, "bedroc", F2)
    panel(axes[0][0], "(A) DOE mean plot — main effects on BEDROC (provisional)")
    axes[0][0].set_ylabel("mean BEDROC")
    block_plot(axes[1], S2B, "bedroc", F2, block="target")   # per-target lines, unblocked by design
    panel(axes[1][0], "(B) Block plot — grey = one target each")
    axes[1][0].set_ylabel("BEDROC by target")
    # rect= reserves top margin for panel()'s label, which is drawn above the axes
    # and excluded from the layout, so tight_layout would leave it no room.
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(FIG_DIR / f"{NB_STEM}_s2_bedroc_mean_block.png", dpi=170)
    plt.savefig(FIG_DIR / f"{NB_STEM}_s2_bedroc_mean_block.pdf")
    plt.show()

    # Same two corrections as the wallclock table: block on dt (the array is not
    # balanced on it) and permute at the CONFIG unit -- the factors are properties of a
    # config, not of a run. An earlier revision fixed only the wallclock response and left
    # this one on the old basis, then broke this cell's indentation so it silently kept
    # shipping the stale CSV (referees, round 9; the indentation was our own).
    E2B = effects_table(S2B, "bedroc", F2, block=["target", "dt_fs"],
                        n_perm=20000, unit="config", strata=S2_STRATA)
    E2B = mark_not_estimable(E2B, S2_ALIAS)
    E2B.round(6).to_csv(DER_DIR / "study2_effects_bedroc.csv", index=False)
    display(E2B.round(4))
    print("permutation p, floor-aware:")
    for _r in E2B.itertuples():
        print(f"  {_r.factor:9s} raw {pfmt(_r.p_perm):>18s}   blocked {pfmt(_r.p_perm_blocked):>18s}")

    # dt is NOT estimable on this response, and its raw p is the only "significant" number
    # in the table. dt=4 survives the rescore on ONE target: count the cells and report the
    # effect with that target removed (referees, round 11).
    _dt4 = S2B[S2B.dt_fs == 4]
    print(f"\n  dt=4 coverage on BEDROC: {len(_dt4)} of {len(S2B)} cells, on "
          f"{_dt4.target.nunique()} target(s): {sorted(_dt4.target.unique())}")
    _drop = S2B[~S2B.target.isin(_dt4.target.unique())]
    print(f"  dt effect on the remaining {_drop.target.nunique()} targets: "
          f"{effect_range(_drop, 'dt_fs', 'bedroc'):.3f} "
          f"(vs {effect_range(S2B, 'dt_fs', 'bedroc'):.3f} on all)")

    _cfg_b = S2B.groupby(["config"] + F2, as_index=False)["bedroc"].mean()
    _avb = anova_config_means(_cfg_b, "bedroc", F2)
    print(f"\n  ANOVA on the {len(_cfg_b)} config means "
          f"(rank {_avb.attrs['rank']}, {_avb.attrs['df_res']} residual df):")
    _avb.round(6).to_csv(DER_DIR / "study2_anova_bedroc.csv", index=False)
    print(_avb.round(4).to_string(index=False))
else:
    print("too few cells clear the gate -- BEDROC DOE analysis withheld")

### 3.8 The measured repeat spread — the only noise floor this design has

**The L27 contains no replicate.** Not one config was run twice under identical settings, so the design cannot estimate its own error term. Every effect above is judged against a permutation reference, not measured reproducibility. An earlier revision called the `dt = 2` configs "physics-identical" repeats. They are not — they differ in `rcoulomb`, `gap` and MTS (multiple-timestep integration factor). That claim is withdrawn.

What the repo *does* contain is an accident: **72 `(config, complex_id)` pairs appear twice** in the wallclock frame, because those runs were retried and both attempts kept. Not a designed replicate — the retried attempt is systematically faster, so the pair mixes run-to-run noise with whatever made the first attempt fail — but it is the only direct measurement of "same job, run again" in the package. It is a *lower* bound on the noise an honest replicate would show. The cell below measures it and puts every Study-2 effect beside it.


In [ ]:
# ---- 3.8 the measured repeat spread, from the 72 duplicated (config, complex_id) keys ----
_dup_keys = wall.duplicated(["config", "complex_id"], keep=False)
_dups = wall[_dup_keys]
_pairs = _dups.groupby(["config", "complex_id"])["wall_s"].agg(["min", "max", "size"])
_pairs["spread_s"] = _pairs["max"] - _pairs["min"]
_MED_SPREAD = float(_pairs.spread_s.median())
print(f"duplicated (config, complex_id) keys: {len(_pairs)}  "
      f"({int(_dup_keys.sum())} rows of {len(wall)})")
print(f"spread between the two attempts, seconds:  "
      f"median {_MED_SPREAD:.0f}   IQR {_pairs.spread_s.quantile(.25):.0f}-"
      f"{_pairs.spread_s.quantile(.75):.0f}   max {_pairs.spread_s.max():.0f}")
print()
print("Every Study-2 wallclock effect against that spread:")
print(f"  {'factor':10s} {'effect (s)':>12s} {'column':>9s} {'x median repeat spread':>24s}")
for _r in E2.sort_values("effect", ascending=False).itertuples():
    _e = _r.effect_blocked if pd.notna(_r.effect_blocked) else _r.effect
    _col = "blocked" if pd.notna(_r.effect_blocked) else "raw"
    print(f"  {_r.factor:10s} {_e:12.1f} {_col:>9s} {_e/_MED_SPREAD:>23.2f}x")
print()
print("READING. Only `dt` (raw, 1360 s) is larger than re-running the same job, and `dt` is")
print("100 % confounded with HMR, so 'dt' here names the timestep-and-mass-repartitioning")
print("pair, not the timestep alone. `mts` at 139 s blocked is 0.4x the repeat spread: it")
print("clears its permutation null and is still smaller than the noise between two runs of")
print("one job. That is the whole Study-2 wallclock result, and it is a screening result:")
print("the production knobs other than the timestep do not move wallclock by as much as")
print("running the same job twice does.")
print()
print("This is a LOWER bound on replicate noise -- retried runs are systematically faster --")
print("so a designed velocity-seed replicate would very likely show more, not less. That")
print("experiment is nb08 section 3.4's 'single most valuable un-run experiment' and it has")
print("not been submitted.")


## 4. Study 2 — the verdict, and the four corrections that produced it

**Result: this L27 screens exactly two factors, and neither is a ranking result.**

| response | what clears its own null | what does not |
|---|---|---|
| wallclock | `dt` (raw, p at the 5.0e-05 floor) and `mts` (raw p = 0.00055, blocked p at the floor) | `rcoulomb` (0.74 / 0.91), `gap` and `nstlist` (raw 0.72 / 0.47; blocked not estimable) |
| BEDROC (α = 20) | **nothing** — the only value at the floor is `dt`, and §3.7 shows it is a coverage artefact | `rcoulomb` 0.22, `nstlist` 0.15, `gap` 0.31, `mts` 0.75 |

A least-squares ANOVA on the 27 config means — the design's own unit, where all five main effects *are* estimable (rank 11, 16 residual df) — agrees on both responses: wallclock `dt` F = 111.7 and `mts` F = 6.31 (p = 0.0095), everything else p ≥ 0.48; BEDROC nothing below p = 0.16. Two independent reference sets, one verdict.

**`dt` on BEDROC is not a result.** `dt = 4` survives the rescore on **one target** (9SI4, 4 of 99 cells). Drop that target and the effect falls from 0.408 to 0.077, against a null q95 of 0.165. Its blocked column is zero by construction because the blocking removes it. The one "significant" number in the BEDROC table is a coverage artefact. Reported here rather than in the table's favour.

**And `mts` is smaller than noise.** §3.8 measures the only repeat data the package contains — 72 `(config, complex_id)` pairs run twice — and `mts`'s blocked effect of 139 s is **0.4× the median spread between two runs of the same job**. It clears its permutation null and is still smaller than re-running the job. Both statements are true, both belong in the conclusion.

### The four corrections, in order, each found by a referee

1. **The exchangeability unit was wrong.** Nulls permuted the factor across 3 115 **runs** for a design that assigns it to 27 **configs**, treating ~115 correlated runs as independent and inflating the null by ~7×. At the config unit, `rcoulomb`, `gap` and `nstlist` all fall from p < 0.02 to p ≈ 0.5 to 0.9.
2. **The permutation group was then wrong.** Permuting the assignment *freely* across configs lets a factor land in alignment with a different factor and absorb its variance. It cost a real result: Study 1's `igb` was withdrawn at p = 0.13 when the design-respecting null puts it **at the permutation floor**. That withdrawal is itself withdrawn.
3. **The fully stratified group does not exist for this array.** Conditioning a factor on the other four leaves **27 strata of size one** — the identity permutation, p = 1.0 for everything, on both responses. A referee measured that by running our own code path before it shipped.
4. **The reference set this design does support is `dt` alone** (3 strata of 9 configs). It preserves the alias structure that matters and leaves the other factors free. `dt` has nothing to condition on and is permuted freely. The numbers above are that null.

### What is aliased, and what is not

Globally the array is sound: `gap × nstlist` is a perfect 3×3 balance over the 27 configs, main effects are orthogonal to 4 × 10⁻¹⁶, design matrix is full rank 11 of 11. An earlier revision said otherwise and was wrong.

The aliasing is **conditional**. Within each `dt` stratum — which is exactly what the blocked column conditions on — the `gap` and `nstlist` partitions of the 9 configs coincide, so each one's blocked main effect is aliased with the other's `dt` interaction. Their blocked columns are therefore `NaN` and marked on the figure. Their **raw** columns stay, because unblocked they are genuinely different marginals (47.9 s vs 71.9 s). "One contrast counted twice" was a looser phrase we used for two rounds; the table's own numbers refute it.

`dt` is separately confounded with **HMR** (hydrogen mass repartitioning): the two share a design column, so "`dt`" names the timestep-and-mass-repartitioning pair throughout, never the timestep alone. Separating them needs a foldover of the L27, which was not run.

### Study 1, by contrast

Study 1 is fully crossed, so its blocked null is **exactly** its raw null — the two columns are identical to the last digit, because at the combo unit blocking is a no-op. That identity is worth stating as the property distinguishing the two studies. It means the "blocking reveals a real effect" narrative this NB once carried has no surviving example in Study 1.

`intdiel` and `igb` both sit at the permutation floor. `surften` (0.077) and `saltcon` (0.090) do not clear. `igb`'s effect is 0.065 BEDROC — the package's own median run-to-run |ΔBEDROC| — so it is detectable and practically nil. Honest pair of statements, which is why we retire it on effect size rather than p-value.

The frame is built per **run** (3 115 rows after the manifest join on status OK). 72 of its `(config, complex_id)` keys appear twice because those runs were retried and both attempts kept. `verify.py` reports how far each effect moves if the fastest attempt is kept instead. `rcoulomb` moves most, by about a third. Run-weighted is the shipped convention. Stated rather than assumed.


In [ ]:
FIGURE_CAPTIONS = {
    's1_effects':
        'NIST DOE step 6 for Study 1: estimated |effects| with their permutation nulls. Raw and target-blocked bars are identical here because the factorial is fully crossed, so at the combo unit blocking is a no-op. intdiel and igb sit at the permutation floor.',
    's1_interactions':
        'NIST DOE step 4 for Study 1: the lower-triangle interaction matrix over the four physics parameters.',
    's1_mean_block':
        'NIST DOE steps 3 and 5 for Study 1: main-effect means per parameter (top) and the block plot with one grey line per target (bottom), which shows whether an effect holds within every target.',
    's1_ordered_scatter':
        'NIST DOE steps 1–2 for Study 1 (the 48-cell physics factorial): ordered data plot and DOE scatter of BEDROC against each of the four parameters.',
    's2_bedroc_mean_block':
        'The same DOE mean and block plots on the BEDROC response, provisional (n≥8 gate, 99 of 243 cells). Note dt = 4 exists on one target only.',
    's2_effects':
        'NIST DOE step 6 for Study 2 on wallclock: |effects| against the config-unit, dt-stratified permutation null. dt is blocked out by its own block and gap/nstlist are marked not estimable, because within each dt stratum their partitions coincide; only MTS has both a bar and a null it clears.',
    's2_interactions':
        'NIST DOE step 4 for Study 2. Panels involving gap × nstlist are not estimable in this resolution-III array conditional on dt.',
    's2_mean_block':
        'NIST DOE steps 3 and 5 for Study 2 on wallclock: main-effect means and the per-target block plot.',
    's2_ordered_scatter':
        'NIST DOE steps 1–2 for Study 2 (the Taguchi L27 production screen): ordered data plot and DOE scatter of wallclock against each of the five parameters.',
}

# ---- captions, keyed by FILENAME ----------------------------------------------------
# The premise printed at the top of every notebook is that suppressed in-figure titles are
# carried by figures/CAPTIONS.md instead. That premise has been false twice: first no caption
# file existed at all, then titles were captured into a list nothing read. The third failure
# was subtler and is fixed here -- the file recorded TITLES but not FILENAMES, so a reader
# holding a PNG could not find its caption, and most notebooks contributed nothing because
# their titles had already been deleted rather than suppressed. Every figure this notebook
# writes now gets a line naming the file; captured titles are appended where they exist.
# verify.py section 16 asserts the coverage, and it is the first check in this package that
# can fail because of a picture (referee, six rounds).
_cap = FIGURES / "CAPTIONS.md"
_mine = sorted(p for p in FIGURES.rglob("*.png") if p.name.startswith(NB_STEM + "_"))
_prev = _cap.read_text() if _cap.exists() else ""
_keep = [l for l in _prev.splitlines()
         if l.startswith("- ") and f"**{NB_STEM}**" not in l]
_lines = []
for _p in _mine:
    _rel = _p.relative_to(FIGURES).as_posix()
    # match on the full basename first, then on the suffix after NB_STEM, because
    # notebooks 02-07 export as {stem}_fig{n} while 08-10 name each figure.
    _d = FIGURE_CAPTIONS.get(_p.name) or FIGURE_CAPTIONS.get(
        _p.stem.removeprefix(NB_STEM + "_"), "")
    _lines.append(f"- `{_rel}` — **{NB_STEM}** — {_d}" if _d
                  else f"- `{_rel}` — **{NB_STEM}** — NO CAPTION WRITTEN")

_lines += [f"- **{NB_STEM}** — suppressed title: {t}" for _, t in _SUPPRESSED_TITLES]
_cap.write_text("# Figure captions\n\nOne line per shipped figure, naming the file, plus any\n"
                "in-figure title suppressed for publication.\n\n"
                + "\n".join(sorted(set(_keep + _lines))) + "\n")
print(f"captions: {len(_mine)} figure(s) and {len(_SUPPRESSED_TITLES)} suppressed title(s) "
      f"recorded in {_cap.name}")
